# Re-evaluate an existing run on the test split

Use this when you want to recompute test-set metrics for a run that's already on Drive
(e.g. after rebuilding the dataset, or if a previous eval was skipped). Skips training.

In [ ]:
REPO_URL    = "https://github.com/tahmid013/yolo.git"
REPO_BRANCH = "main"
DATASET_VERSION = "v1"
RUN_NAME    = "yolo11s_20260511-142233"   # name of the Drive run folder to re-eval

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!nvidia-smi

In [ ]:
import os, shutil
if os.path.isdir('/content/code'):
    shutil.rmtree('/content/code')
!git clone --branch {REPO_BRANCH} {REPO_URL} /content/code
%cd /content/code
!pip install -q -r requirements.txt

In [ ]:
from pipeline.dataset import ensure_dataset
from pipeline import paths
data_yaml = ensure_dataset(
    zip_path=paths.dataset_zip(DATASET_VERSION),
    meta_path=paths.dataset_meta(DATASET_VERSION),
    target=paths.LOCAL_DATASET,
)

In [ ]:
# Copy the run from Drive to local for fast file I/O during eval, then push back.
import shutil
from pathlib import Path
src = paths.RUNS_DIR / RUN_NAME
dst = paths.LOCAL_RUNS / RUN_NAME
if dst.exists():
    shutil.rmtree(dst)
shutil.copytree(src, dst)
print('local run dir:', dst)

In [ ]:
from pipeline.evaluate import run as eval_run
eval_run(run_dir=dst, data_yaml=data_yaml, drive_runs_dir=paths.RUNS_DIR)